In [3]:
import pandas as pd
import yfinance as yf

In [4]:
#dictionary of global indices
indices = {
    "^GSPC": "S&P 500",
    "^IXIC": "NASDAQ",
    "^DJI": "Dow Jones",
    "^NSEI": "Nifty 50",
    "^BSESN": "Sensex"
}

In [5]:
import pandas as pd
import yfinance as yf

# Dictionary of global indices
indices = {
    "^GSPC": "S&P 500",
    "^IXIC": "NASDAQ",
    "^DJI": "Dow Jones",
    "^NSEI": "Nifty 50",
    "^BSESN": "Sensex"
}

all_data = []

for ticker, index_name in indices.items():

    # Read latest available TradeDate from SQL
    query = f"""
    SELECT MAX(TradeDate) AS LastTradeDate
    FROM GlobalIndexPrices
    WHERE Ticker = '{ticker}'
    """

    last_date = pd.read_sql(query, engine).iloc[0, 0]

    # If no data exists
    if pd.isna(last_date):
        start_date = "2020-01-01"
    else:
        start_date = (pd.to_datetime(last_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    print(f"Updating {index_name} from {start_date}")

    df = yf.download(
        ticker,
        start=start_date,
        progress=False,
        auto_adjust=False
    )

    if df.empty:
        print(f"No new data for {index_name}")
        continue

    # Flatten MultiIndex
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns.name = None

    df.reset_index(inplace=True)

    df["Ticker"] = ticker
    df["IndexName"] = index_name

    df = df[
        [
            "Date",
            "Ticker",
            "IndexName",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ]
    ]

    all_data.append(df)

Updating S&P 500 from 2026-07-31
Updating NASDAQ from 2026-07-31
Updating Dow Jones from 2026-07-31
Updating Nifty 50 from 2026-08-01
Updating Sensex from 2026-08-01


In [6]:
#validation
global_index_df = pd.concat(all_data, ignore_index=True)

In [7]:
#Shape
print("\n Shape")
print(global_index_df.shape)


 Shape
(10, 8)


In [8]:
#datatypes
print("\nDatatypes")
print(global_index_df.dtypes)


Datatypes
Date         datetime64[ns]
Ticker               object
IndexName            object
Open                float64
High                float64
Low                 float64
Close               float64
Volume                int64
dtype: object


In [9]:
#checking the missing values
print("\nMissing Values:")
print(global_index_df.isnull().sum())


Missing Values:
Date         0
Ticker       0
IndexName    0
Open         0
High         0
Low          0
Close        0
Volume       0
dtype: int64


In [10]:
#checking the duplicat rows
print("\nDuplicated Values")
print(global_index_df.duplicated().sum())


Duplicated Values
0


In [11]:
global_index_df.tail(100)

,Date,Ticker,IndexName,Open,High,Low,Close,Volume
0,2026-07-31,^GSPC,S&P 500,7462.129883,7512.040039,7399.830078,7489.720215,5391020000
1,2026-08-03,^GSPC,S&P 500,7504.779785,7610.040039,7504.779785,7600.500000,5201470000
2,2026-07-31,^IXIC,NASDAQ,25340.710938,25460.859375,25004.330078,25373.849609,11942500000
3,2026-08-03,^IXIC,NASDAQ,25452.660156,25967.439453,25420.349609,25913.900391,11118460000
4,2026-07-31,^DJI,Dow Jones,52235.031250,52623.140625,51996.320312,52485.031250,697510000
5,2026-08-03,^DJI,Dow Jones,52759.058594,53230.078125,52759.058594,53178.410156,60490000
6,2026-08-03,^NSEI,Nifty 50,24572.699219,24774.300781,24515.150391,24774.300781,342300
7,2026-08-04,^NSEI,Nifty 50,24703.900391,24703.900391,24427.949219,24614.900391,0
8,2026-08-03,^BSESN,Sensex,78883.343750,78895.101562,78497.343750,78639.031250,21000
9,2026-08-04,^BSESN,Sensex,79132.968750,79143.148438,78211.867188,78428.953125,0


In [12]:
global_index_df.rename(
    columns={
        "Date": "TradeDate",
        "Open": "OpenPrice",
        "High": "HighPrice",
        "Low": "LowPrice",
        "Close": "ClosePrice",
    },
    inplace=True,
)

In [13]:
print(global_index_df.columns)

Index(['TradeDate', 'Ticker', 'IndexName', 'OpenPrice', 'HighPrice',
       'LowPrice', 'ClosePrice', 'Volume'],
      dtype='object')


In [14]:
from sqlalchemy import create_engine
import urllib

In [15]:
from sqlalchemy import create_engine
import urllib

server = "ASTHA"
database = "GlobalIndexAnalytics"

connection_string = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

connection_url = (
    "mssql+pyodbc:///?odbc_connect="
    + urllib.parse.quote_plus(connection_string)
)

engine = create_engine(connection_url)

In [16]:
with engine.connect() as conn:
    print("Connection Successful!")

Connection Successful!


C:\Users\nagat\AppData\Local\Temp\ipykernel_44416\2754817518.py:1: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


In [17]:
global_index_df.to_sql(
    name="GlobalIndexPrices",
    con=engine,
    if_exists="append",
    index=False,
)

10